In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
from google.colab import files
files = files.upload()
NVDA = pd.read_csv("NVDA.csv")

Saving NVDA.csv to NVDA.csv


In [4]:
NVDA['MA_5'] = NVDA['Close'].rolling(5).mean()
NVDA['MA_10'] = NVDA['Close'].rolling(10).mean()
NVDA['MA_20'] = NVDA['Close'].rolling(20).mean()

In [5]:
delta = NVDA['Close'].diff()

gains = delta.clip(lower=0)
losses = -delta.clip(upper=0)

avg_gains = gains.rolling(14).mean()
avg_losses = losses.rolling(14).mean()

RS = (avg_gains / avg_losses)
NVDA["RSI_14"] = 100 - (100 / (1 + RS))

In [6]:
EMA_12 = NVDA['Close'].ewm(span=12,adjust=False).mean()
EMA_26 = NVDA['Close'].ewm(span=26,adjust=False).mean()
NVDA['MACD'] = EMA_12-EMA_26
NVDA['MACD_Signal'] = NVDA['MACD'].ewm(span=9,adjust=False).mean()
NVDA['MACD_Histogram'] = NVDA['MACD_Signal'] - NVDA['MACD']
NVDA = NVDA.dropna()

In [7]:
X = NVDA.drop(columns=['Close','Date']).iloc[:-1].values
y = NVDA['Close'].iloc[1:].values
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)


In [9]:
from sklearn.linear_model import LinearRegression
Linear_model = LinearRegression()
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train_scaler)
X_test_poly = poly.transform(X_test_scaler)
Linear_model.fit(X_poly_train,y_train)
from sklearn import linear_model
Linear_model_lasso = linear_model.Lasso(alpha = 0.1, max_iter = 100, tol = 0.01)
Linear_model_lasso.fit(X_poly_train,y_train)
y_train_pred = Linear_model_lasso.predict(X_poly_train)
y_test_pred = Linear_model_lasso.predict(X_test_poly)
train_r2 = r2_score(y_train,y_train_pred)
test_r2 = r2_score(y_test,y_test_pred)
print("R2 for training data:", round(train_r2,4))
print("R2 for testing data:", round(test_r2,4))

R2 for training data: 0.995
R2 for testing data: 0.9945


In [11]:
NVDA['Date'] = pd.to_datetime(NVDA['Date'])
dates = NVDA['Date'].iloc[-len(y_test):]
graph_of_model = px.scatter(x=dates,y=y_test)
graph_of_model.add_scatter(x=dates,y=y_test_pred,mode='lines',name='Prediction Line')
graph_of_model.show()

In [12]:
from sklearn.metrics import r2_score
model_accuracy = r2_score(y_test,y_test_pred)
model_accuracy=model_accuracy*100
print("The accuracy of this model is",round(model_accuracy,2),"%")

The accuracy of this model is 99.45 %


In [13]:
X_next_day = NVDA[['Open','High','Low','Volume','MA_5','MA_10','MA_20','RSI_14','MACD','MACD_Signal','MACD_Histogram']].iloc[-1:].values
X_next_day_scaled = scaler.transform(X_next_day)
X_next_day_poly = poly.transform(X_next_day_scaled)
next_day_close = Linear_model_lasso.predict(X_next_day_poly)
print("The model's prediction for the next trading day(December 8 2025) is $",next_day_close.round(2))

The model's prediction for the next trading day(December 8 2025) is $ [229.07]
